# Unit 2: Q-Learning with FrozenLake-v1 ⛄ and Taxi-v3 🚕

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/thumbnail.jpg" alt="Unit 2 Thumbnail">

In this notebook, **you'll code your first Reinforcement Learning agent from scratch** to play FrozenLake ❄️ using Q-Learning, share it with the community, and experiment with different configurations.

⬇️ Here is an example of what **you will achieve in just a couple of minutes.** ⬇️


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/envs.gif" alt="Environments"/>

###🎮 Environments:

- [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
- [Taxi-v3](https://gymnasium.farama.org/environments/toy_text/taxi/)

###📚 RL-Library:

- Python and NumPy
- [Gymnasium](https://gymnasium.farama.org/)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives

Use Gymnasium, implement tabular Q-learning, train FrozenLake and Taxi agents, save local replay videos, and publish the trained Q-tables to Hugging Face.

## This notebook is from the Deep Reinforcement Learning Course

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

And more check 📚 the syllabus 👉 https://simoninithomas.github.io/deep-rl-course

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️

Before diving into the notebook, you need to:

🔲 📚 **Study [Q-Learning by reading Unit 2](https://huggingface.co/deep-rl-course/unit2/introduction)**  🤗  

## A small recap of Q-Learning

*Q-Learning* **is the RL algorithm that**:

- Trains *Q-Function*, an **action-value function** that encoded, in internal memory, by a *Q-table* **that contains all the state-action pair values.**

- Given a state and action, our Q-Function **will search the Q-table for the corresponding value.**
    
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-function-2.jpg" alt="Q function"  width="100%"/>

- When the training is done,**we have an optimal Q-Function, so an optimal Q-Table.**
    
- And if we **have an optimal Q-function**, we
have an optimal policy, since we **know for, each state, the best action to take.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/link-value-policy.jpg" alt="Link value policy"  width="100%"/>


But, in the beginning, our **Q-Table is useless since it gives arbitrary value for each state-action pair (most of the time we initialize the Q-Table to 0 values)**. But, as we’ll explore the environment and update our Q-Table it will give us better and better approximations

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/q-learning.jpeg" alt="q-learning.jpeg" width="100%"/>

This is the Q-Learning pseudocode:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


# Let's code our first Reinforcement Learning algorithm 🚀

To validate this hands-on for the [certification process](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process), you need to push your trained Taxi model to the Hub and **get a result of >= 4.5**.

To find your result, go to the [leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) and find your model, **the result = mean_reward - std of reward**

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## Local setup

Install `requirements.txt` once, then run this notebook with Python from the `Notebook/notebooks` directory. RGB array rendering works headlessly.

In [ ]:
print('Dependencies are listed in Unit 2/requirements.txt and installed before notebook execution.')

In [ ]:
print('Gymnasium rgb_array rendering works headlessly; no apt packages or virtual display are required.')

The local kernel remains active throughout training.

In [ ]:
print('Local kernel ready; no Colab runtime restart is needed.')

In [ ]:
import os
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
print('SDL_VIDEODRIVER =', os.environ['SDL_VIDEODRIVER'])

## Import the packages 📦

In addition to the installed libraries, we also use:

- `random`: To generate random numbers (that will be useful for epsilon-greedy policy).
- `imageio`: To generate a replay video.

In [ ]:
from pathlib import Path
import json, pickle, random, shutil
import numpy as np
import gymnasium as gym
import imageio.v2 as imageio
from PIL import Image, ImageDraw
from IPython.display import display, Video
random.seed(42)
np.random.seed(42)
RUNS = Path('../runs/1m').resolve()
for name in ('models', 'videos', 'hub'):
    (RUNS / name).mkdir(parents=True, exist_ok=True)
print('Gymnasium', gym.__version__, '| outputs:', RUNS)

We're now ready to code our Q-Learning algorithm 🔥

# Part 1: Frozen Lake ⛄ (non slippery version)

## Create and understand [FrozenLake environment ⛄]((https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
---

💡 A good habit when you start to use an environment is to check its documentation

👉 https://gymnasium.farama.org/environments/toy_text/frozen_lake/

---

We're going to train our Q-Learning agent **to navigate from the starting state (S) to the goal state (G) by walking only on frozen tiles (F) and avoid holes (H)**.

We can have two sizes of environment:

- `map_name="4x4"`: a 4x4 grid version
- `map_name="8x8"`: a 8x8 grid version


The environment has two modes:

- `is_slippery=False`: The agent always moves **in the intended direction** due to the non-slippery nature of the frozen lake (deterministic).
- `is_slippery=True`: The agent **may not always move in the intended direction** due to the slippery nature of the frozen lake (stochastic).

For now let's keep it simple with the 4x4 map and non-slippery.
We add a parameter called `render_mode` that specifies how the environment should be visualised. In our case because we **want to record a video of the environment at the end, we need to set render_mode to rgb_array**.

As [explained in the documentation](https://gymnasium.farama.org/api/env/#gymnasium.Env.render) “rgb_array”: Return a single frame representing the current state of the environment. A frame is a np.ndarray with shape (x, y, 3) representing RGB values for an x-by-y pixel image.

In [ ]:
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode='rgb_array')
env.reset(seed=42)
env.action_space.seed(42)
print('FrozenLake environment:', env.spec.id, env.spec.kwargs)

### Solution

In [ ]:
print('Solution: the 4x4 non-slippery FrozenLake environment is ready.')

You can create your own custom grid like this:

```python
desc=["SFFF", "FHFH", "FFFH", "HFFG"]
gym.make('FrozenLake-v1', desc=desc, is_slippery=True)
```

but we'll use the default environment for now.

### Let's see what the Environment looks like:


In [ ]:
print('Observation space:', env.observation_space, '| sample:', env.observation_space.sample())

We see with `Observation Space Shape Discrete(16)` that the observation is an integer representing the **agent’s current position as current_row * ncols + current_col (where both the row and col start at 0)**.

For example, the goal position in the 4x4 map can be calculated as follows: 3 * 4 + 3 = 15. The number of possible observations is dependent on the size of the map. **For example, the 4x4 map has 16 possible observations.**


For instance, this is what state = 0 looks like:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">

In [ ]:
print('Action space:', env.action_space, '| sample:', env.action_space.sample())

The action space (the set of possible actions the agent can take) is discrete with 4 actions available 🎮:
- 0: GO LEFT
- 1: GO DOWN
- 2: GO RIGHT
- 3: GO UP

Reward function 💰:
- Reach goal: +1
- Reach hole: 0
- Reach frozen: 0

## Create and Initialize the Q-table 🗄️

(👀 Step 1 of the pseudocode)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


It's time to initialize our Q-table! To know how many rows (states) and columns (actions) to use, we need to know the action and observation space. We already know their values from before, but we'll want to obtain them programmatically so that our algorithm generalizes for different environments. Gym provides us a way to do that: `env.action_space.n` and `env.observation_space.n`


In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n
print('States:', state_space, '| actions:', action_space)

In [ ]:
def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space), dtype=np.float64)
print('Initialized Q-table shape:', initialize_q_table(state_space, action_space).shape)

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)
print('FrozenLake Q-table initialized:', Qtable_frozenlake.shape)

### Solution

In [ ]:
state_space, action_space = env.observation_space.n, env.action_space.n
print('Solution: states =', state_space, ', actions =', action_space)

In [ ]:
print('Solution: np.zeros((state_space, action_space)) gives', initialize_q_table(state_space, action_space).shape)

In [ ]:
print('FrozenLake initial nonzero Q-values:', np.count_nonzero(Qtable_frozenlake))

## Define the greedy policy 🤖

Remember we have two policies since Q-Learning is an **off-policy** algorithm. This means we're using a **different policy for acting and updating the value function**.

- Epsilon-greedy policy (acting policy)
- Greedy-policy (updating policy)

The greedy policy will also be the final policy we'll have when the Q-learning agent completes training. The greedy policy is used to select an action using the Q-table.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/off-on-4.jpg" alt="Q-Learning" width="100%"/>


In [ ]:
def greedy_policy(Qtable, state):
    return int(np.argmax(Qtable[state]))
print('Greedy action for initial state:', greedy_policy(Qtable_frozenlake, 0))

#### Solution

In [ ]:
print('Solution: choose argmax of the current state row:', greedy_policy(Qtable_frozenlake, 0))

##Define the epsilon-greedy policy 🤖

Epsilon-greedy is the training policy that handles the exploration/exploitation trade-off.

The idea with epsilon-greedy:

- With *probability 1 - ɛ* : **we do exploitation** (i.e. our agent selects the action with the highest state-action pair value).

- With *probability ɛ*: we do **exploration** (trying a random action).

As the training continues, we progressively **reduce the epsilon value since we will need less and less exploration and more exploitation.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-4.jpg" alt="Q-Learning" width="100%"/>


In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon, action_space):
    return int(action_space.sample()) if random.random() < epsilon else greedy_policy(Qtable, state)
print('Exploratory action:', epsilon_greedy_policy(Qtable_frozenlake, 0, 1.0, env.action_space))

#### Solution

In [ ]:
print('Solution: epsilon-greedy policy accepts epsilon and the environment action space.')

## Define the hyperparameters ⚙️

The exploration related hyperparamters are some of the most important ones.

- We need to make sure that our agent **explores enough of the state space** to learn a good value approximation. To do that, we need to have progressive decay of the epsilon.
- If you decrease epsilon too fast (too high decay_rate), **you take the risk that your agent will be stuck**, since your agent didn't explore enough of the state space and hence can't solve the problem.

In [ ]:
training_steps = 1_000_000
learning_rate = 0.7
gamma = 0.95
n_eval_episodes = 100
env_id = 'FrozenLake-v1'
max_steps = 99
eval_seed = list(range(100))
max_epsilon, min_epsilon, decay_rate = 1.0, 0.05, 0.0005
print('FrozenLake training budget:', training_steps, 'environment steps')

## Create the training loop method

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

The training loop goes like this:

```
For episode in the total of training episodes:

Reduce epsilon (since we need less and less exploration)
Reset the environment

  For step in max timesteps:    
    Choose the action At using epsilon greedy policy
    Take the action (a) and observe the outcome state(s') and reward (r)
    Update the Q-value Q(s,a) using Bellman equation Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
    If done, finish the episode
    Our next state is the new state
```

In [ ]:
def train(training_steps, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable, learning_rate, gamma):
    steps = episodes = 0
    next_report = 200_000
    episode_returns = []
    while steps < training_steps:
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episodes)
        state, _ = env.reset()
        episode_reward = 0.0
        for _ in range(min(max_steps, training_steps - steps)):
            action = epsilon_greedy_policy(Qtable, state, epsilon, env.action_space)
            next_state, reward, terminated, truncated, _ = env.step(action)
            target = reward if (terminated or truncated) else reward + gamma * np.max(Qtable[next_state])
            Qtable[state, action] += learning_rate * (target - Qtable[state, action])
            state = next_state
            episode_reward += reward
            steps += 1
            if terminated or truncated:
                break
        episodes += 1
        episode_returns.append(episode_reward)
        if steps >= next_report:
            print(f'{env.spec.id}: {steps:,} steps, {episodes:,} episodes, recent mean reward {np.mean(episode_returns[-100:]):.2f}')
            next_report += 200_000
    return Qtable, {'training_steps': steps, 'n_training_episodes': episodes, 'last_100_train_reward': float(np.mean(episode_returns[-100:]))}
print('Training function ready.')

#### Solution

In [ ]:
print('Solution: the training loop updates Q(s,a) after each environment step and stops at the exact step budget.')

## Train the Q-Learning agent 🏃

In [ ]:
Qtable_frozenlake, frozen_stats = train(training_steps, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake, learning_rate, gamma)
print('FrozenLake training:', frozen_stats)

## Let's see what our Q-Learning table looks like now 👀

In [ ]:
print('FrozenLake Q-table:\n', Qtable_frozenlake)

## The evaluation method 📝

- We defined the evaluation method that we're going to use to test our Q-Learning agent.

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seeds):
    rewards = []
    for episode in range(n_eval_episodes):
        state, _ = env.reset(seed=seeds[episode] if seeds else episode)
        total_reward = 0.0
        for _ in range(max_steps):
            action = greedy_policy(Q, state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break
        rewards.append(total_reward)
    return float(np.mean(rewards)), float(np.std(rewards))
print('Evaluation function ready.')

## Evaluate our Q-Learning agent 📈

- Usually, you should have a mean reward of 1.0
- The **environment is relatively easy** since the state space is really small (16). What you can try to do is [to replace it with the slippery version](https://gymnasium.farama.org/environments/toy_text/frozen_lake/), which introduces stochasticity, making the environment more complex.

In [ ]:
frozen_mean, frozen_std = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f'FrozenLake: {frozen_mean:.2f} +/- {frozen_std:.2f} over {n_eval_episodes} episodes')

## Save the trained agents

The following cells save local model packages and replay videos. The upload script publishes the packages after the notebook finishes.

Under the hood, the Hub uses git-based repositories (don't worry if you don't know what git is), which means you can update the model with new versions as you experiment and improve your agent.

Define the local model packaging helpers.

In [ ]:
from datetime import datetime, timezone
print('Local package dependencies loaded.')

In [ ]:
def _episode_frames(env, Qtable, seeds, max_steps):
    frames, rewards = [], []
    for seed in seeds:
        state, _ = env.reset(seed=seed)
        frames.append(np.asarray(env.render()))
        total = 0.0
        for _ in range(max_steps):
            state, reward, terminated, truncated, _ = env.step(greedy_policy(Qtable, state))
            frames.append(np.asarray(env.render()))
            total += reward
            if terminated or truncated:
                break
        rewards.append(total)
    return frames, rewards

def record_video(env, Qtable, path, seeds=(16, 54, 165), max_steps=99, fps=4):
    frames, rewards = _episode_frames(env, Qtable, seeds, max_steps)
    with imageio.get_writer(path, fps=fps, codec='libx264', macro_block_size=2) as writer:
        for frame in frames:
            writer.append_data(frame)
    return {'path': str(path), 'frames': len(frames), 'rewards': rewards}
print('Video recorder ready.')

In [ ]:
def package_model(model, env, repo_name, mean_reward, std_reward, video_path):
    package = RUNS / 'hub' / repo_name
    package.mkdir(parents=True, exist_ok=True)
    with (package / 'q-learning.pkl').open('wb') as f:
        pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)
    results = {'env_id': model['env_id'], 'mean_reward': mean_reward, 'std_reward': std_reward, 'n_eval_episodes': model['n_eval_episodes'], 'eval_datetime': datetime.now(timezone.utc).isoformat()}
    (package / 'results.json').write_text(json.dumps(results, indent=2) + '\n')
    env_kwargs = {k: model[k] for k in ('map_name', 'is_slippery') if k in model}
    card = f"""---\ntags:\n- reinforcement-learning\n- q-learning\n- {model['env_id']}\n---\n# Q-learning agent for {model['env_id']}\n\nTrained for {model['training_steps']:,} environment steps with tabular Q-learning.\nMean reward over {model['n_eval_episodes']} evaluation episodes: {mean_reward:.2f} ± {std_reward:.2f}.\n\nLoad `q-learning.pkl` with Python pickle from a trusted source. Create the environment with `gym.make({model['env_id']!r}, **{env_kwargs!r})` and act with `np.argmax(model['qtable'][state])`.\n"""
    (package / 'README.md').write_text(card)
    shutil.copy2(video_path, package / 'replay.mp4')
    return package
print('Model packaging function ready.')

The package helper saves the Q-table, evaluation result, model card, and replay video locally. The upload script publishes only the Q-table model file.

To publish after training, use `python Notebook/scripts/upload_to_hub.py` with a Hugging Face login or `HF_TOKEN`.

In [ ]:
print('Hugging Face upload runs after local training via Notebook/scripts/upload_to_hub.py.')

The upload script checks that the authenticated account matches the Unit 1 account, `tashobi02`.

Store the FrozenLake hyperparameters and Q-table in a model dictionary.

In [ ]:
frozen_model = {'env_id': env_id, 'map_name': '4x4', 'is_slippery': False, 'max_steps': max_steps, 'training_steps': frozen_stats['training_steps'], 'n_training_episodes': frozen_stats['n_training_episodes'], 'n_eval_episodes': n_eval_episodes, 'eval_seed': eval_seed, 'learning_rate': learning_rate, 'gamma': gamma, 'max_epsilon': max_epsilon, 'min_epsilon': min_epsilon, 'decay_rate': decay_rate, 'qtable': Qtable_frozenlake}
print('FrozenLake model keys:', sorted(frozen_model))

Save a local FrozenLake model package and replay video.

In [ ]:
print('FrozenLake model Q-table shape:', frozen_model['qtable'].shape)

In [ ]:
frozen_video = RUNS / 'videos/frozenlake-1m.mp4'
frozen_video_info = record_video(env, Qtable_frozenlake, frozen_video, seeds=(16, 54, 165), max_steps=max_steps)
frozen_package = package_model(frozen_model, env, 'q-FrozenLake-v1-4x4-noSlippery-1m', frozen_mean, frozen_std, frozen_video)
shutil.copy2(frozen_package / 'q-learning.pkl', RUNS / 'models/frozenlake-1m.pkl')
print('FrozenLake replay:', frozen_video_info)
print('FrozenLake Hub package:', frozen_package)

FrozenLake is saved. Next, train a second Q-learning agent in Taxi.

# Part 2: Taxi-v3 🚖

## Create and understand [Taxi-v3 🚕](https://gymnasium.farama.org/environments/toy_text/taxi/)
---

💡 A good habit when you start to use an environment is to check its documentation

👉 https://gymnasium.farama.org/environments/toy_text/taxi/

---

In `Taxi-v3` 🚕, there are four designated locations in the grid world indicated by R(ed), G(reen), Y(ellow), and B(lue).

When the episode starts, **the taxi starts off at a random square** and the passenger is at a random location. The taxi drives to the passenger’s location, **picks up the passenger**, drives to the passenger’s destination (another one of the four specified locations), and then **drops off the passenger**. Once the passenger is dropped off, the episode ends.


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">


In [ ]:
env.close()
env = gym.make('Taxi-v3', render_mode='rgb_array')
env.reset(seed=42)
env.action_space.seed(42)
print('Taxi environment:', env.spec.id)

There are **500 discrete states since there are 25 taxi positions, 5 possible locations of the passenger** (including the case when the passenger is in the taxi), and **4 destination locations.**


In [ ]:
state_space = env.observation_space.n
print('Taxi states:', state_space)

In [ ]:
action_space = env.action_space.n
print('Taxi actions:', action_space)

The action space (the set of possible actions the agent can take) is discrete with **6 actions available 🎮**:

- 0: move south
- 1: move north
- 2: move east
- 3: move west
- 4: pickup passenger
- 5: drop off passenger

Reward function 💰:

- -1 per step unless other reward is triggered.
- +20 delivering passenger.
- -10 executing “pickup” and “drop-off” actions illegally.

In [ ]:
Qtable_taxi = initialize_q_table(state_space, action_space)
print('Taxi Q-table shape:', Qtable_taxi.shape)

## Define the hyperparameters ⚙️

⚠ DO NOT MODIFY EVAL_SEED: the eval_seed array **allows us to evaluate your agent with the same taxi starting positions for every classmate**

In [ ]:
training_steps = 1_000_000
learning_rate = 0.7
n_eval_episodes = 100
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148]
env_id = 'Taxi-v3'
max_steps = 99
gamma = 0.95
max_epsilon, min_epsilon, decay_rate = 1.0, 0.05, 0.005
print('Taxi training budget:', training_steps, 'environment steps')

## Train our Q-Learning agent 🏃

In [ ]:
Qtable_taxi, taxi_stats = train(training_steps, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi, learning_rate, gamma)
print('Taxi training:', taxi_stats)
taxi_mean, taxi_std = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed)
print(f'Taxi: {taxi_mean:.2f} +/- {taxi_std:.2f} over {n_eval_episodes} episodes')

## Save the Taxi model

Store the Taxi hyperparameters and Q-table before packaging.

In [ ]:
taxi_model = {'env_id': env_id, 'max_steps': max_steps, 'training_steps': taxi_stats['training_steps'], 'n_training_episodes': taxi_stats['n_training_episodes'], 'n_eval_episodes': n_eval_episodes, 'eval_seed': eval_seed, 'learning_rate': learning_rate, 'gamma': gamma, 'max_epsilon': max_epsilon, 'min_epsilon': min_epsilon, 'decay_rate': decay_rate, 'qtable': Qtable_taxi}
print('Taxi model Q-table shape:', taxi_model['qtable'].shape)

In [ ]:
taxi_video = RUNS / 'videos/taxi-1m.mp4'
taxi_video_info = record_video(env, Qtable_taxi, taxi_video, seeds=(16, 54, 165), max_steps=max_steps)
taxi_package = package_model(taxi_model, env, 'q-Taxi-v3-1m', taxi_mean, taxi_std, taxi_video)
shutil.copy2(taxi_package / 'q-learning.pkl', RUNS / 'models/taxi-1m.pkl')
print('Taxi replay:', taxi_video_info)
print('Taxi Hub package:', taxi_package)

The Taxi model and replay are saved locally. The upload script publishes both agents after the notebook run.

# Part 3: Reload saved local models

Check that both saved Q-tables evaluate correctly and render a joint demonstration.

The local package uses the same `q-learning.pkl` filename as the Hub model.

Define a local loader and verify the saved models.

In [ ]:
def load_local_model(package):
    with (package / 'q-learning.pkl').open('rb') as f:
        return pickle.load(f)
print('Local model loader ready.')

Reload the Taxi model, then the FrozenLake model, and create the combined demonstration video.

In [ ]:
loaded_taxi = load_local_model(taxi_package)
loaded_taxi_mean, loaded_taxi_std = evaluate_agent(env, loaded_taxi['max_steps'], loaded_taxi['n_eval_episodes'], loaded_taxi['qtable'], loaded_taxi['eval_seed'])
print(f'Reloaded Taxi: {loaded_taxi_mean:.2f} +/- {loaded_taxi_std:.2f}')

In [ ]:
loaded_frozen = load_local_model(frozen_package)
frozen_env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode='rgb_array')
loaded_frozen_mean, loaded_frozen_std = evaluate_agent(frozen_env, loaded_frozen['max_steps'], loaded_frozen['n_eval_episodes'], loaded_frozen['qtable'], loaded_frozen['eval_seed'])
print(f'Reloaded FrozenLake: {loaded_frozen_mean:.2f} +/- {loaded_frozen_std:.2f}')
from PIL import ImageOps
left_frames, _ = _episode_frames(frozen_env, loaded_frozen['qtable'], (16,54,165), 99)
right_frames, _ = _episode_frames(env, loaded_taxi['qtable'], (16,54,165), 99)
combined_path = RUNS / 'videos/all-agents-1m.mp4'
count = max(len(left_frames), len(right_frames))
with imageio.get_writer(combined_path, fps=4, codec='libx264', macro_block_size=2) as writer:
    for index in range(count):
        canvas = Image.new('RGB', (1040, 600), 'white')
        left = ImageOps.contain(Image.fromarray(left_frames[min(index, len(left_frames)-1)]), (500, 530))
        right = ImageOps.contain(Image.fromarray(right_frames[min(index, len(right_frames)-1)]), (500, 530))
        canvas.paste(left, ((520-left.width)//2, 50+(530-left.height)//2))
        canvas.paste(right, (520+(520-right.width)//2, 50+(530-right.height)//2))
        draw = ImageDraw.Draw(canvas)
        draw.text((20, 20), 'FrozenLake Q-learning agent', fill='black')
        draw.text((540, 20), 'Taxi Q-learning agent', fill='black')
        writer.append_data(np.asarray(canvas))
print('Combined demonstration:', combined_path, '| frames:', count)
display(Video(str(combined_path), embed=True))
frozen_env.close()
env.close()

## Some additional challenges 🏆

The best way to learn **is to try things on your own**! As you saw, the current agent is not doing great. As a first suggestion, you can train for more steps. With 1,000,000 steps, we saw some great results!

In the [Leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) you will find your agents. Can you get to the top?

Here are some ideas to climb up the leaderboard:

* Train more steps
* Try different hyperparameters by looking at what your classmates have done.
* **Push your new trained model** on the Hub 🔥

Are walking on ice and driving taxis too boring to you? Try to **change the environment**, why not use FrozenLake-v1 slippery version? Check how they work [using the gymnasium documentation](https://gymnasium.farama.org/) and have fun 🎉.

_____________________________________________________________________
Congrats 🥳, you've just implemented, trained, and uploaded your first Reinforcement Learning agent.

Understanding Q-Learning is an **important step to understanding value-based methods.**

In the next Unit with Deep Q-Learning, we'll see that while creating and updating a Q-table was a good strategy — **however, it is not scalable.**

For instance, imagine you create an agent that learns to play Doom.

<img src="https://vizdoom.cs.put.edu.pl/user/pages/01.tutorial/basic.png" alt="Doom"/>

Doom is a large environment with a huge state space (millions of different states). Creating and updating a Q-table for that environment would not be efficient.

That's why we'll study Deep Q-Learning in the next unit, an algorithm **where we use a neural network that approximates, given a state, the different Q-values for each action.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>


See you in Unit 3! 🔥

## Keep learning, stay awesome 🤗